# 03 — Average True Range (ATR)

## Goal
Compute ATR(14) — a **volatility ruler** that makes every threshold adaptive to the current market conditions.  
Instead of hardcoding "2 points is a big move", we say "1.5× ATR is a big move" — so the same algorithm works whether ATR = 0.5 or ATR = 5.0.

## Equations

**True Range (TR)** — the largest of three distances:
$$\text{TR}_i = \max\!\Bigl(\text{high}_i - \text{low}_i,\;\bigl|\text{high}_i - \text{close}_{i-1}\bigr|,\;\bigl|\text{low}_i - \text{close}_{i-1}\bigr|\Bigr)$$

The three terms capture:
1. The candle's own span
2. A gap-up (today's high vs yesterday's close)
3. A gap-down (today's low vs yesterday's close)

**ATR — Wilder's Smoothed Moving Average:**
$$\text{ATR}_i = \frac{\text{ATR}_{i-1} \times (n-1) + \text{TR}_i}{n}, \quad n = 14$$

This is an **exponential decay** formula — older candles contribute less, but they never fully disappear.

## Why It Matters
| Usage | Formula |
|-------|---------|
| Base tightness gate | $\dfrac{\text{base\_width}}{\text{avg\_ATR}} \leq 2.5$ |
| Per-candle range gate | $\dfrac{\text{candle\_range}}{\text{ATR}} \leq 1.5$ |
| Leg strength gate | $\text{leg\_net} \geq 1.5 \times \text{ATR}$ |
| Departure gate | $\text{departure} \geq 1.5 \times \text{ATR}$ |

## Visualization
The upper panel shows price; the lower panel shows the ATR(14) line.  
Notice ATR rises during fast-moving scenarios (e.g. Scenario D) and stays low during tight bases (e.g. Scenario A).


In [5]:
import pandas as pd

df = pd.read_csv("../fixtures_enhanced.csv", index_col=0, parse_dates=True)
df["true-range"] = df[["high", "low", "prev-close"]].apply(lambda x: max(x["high"] - x["low"], abs(x["high"] - x["prev-close"]), abs(x["low"] - x["prev-close"])), axis=1)
df = df.drop(columns=["prev-close"])

n = 14
tr = df["true-range"].to_numpy()
atr = tr.copy()
for i in range(1, len(tr)):
    atr[i] = (atr[i-1] * (n - 1) + tr[i]) / n
df["atr"] = atr
df = df.drop(columns=["true-range"])
df.to_csv("../fixtures_enhanced.csv")
df.head(20)


,open,high,low,close,volume,range,body,body-ratio,is_base,is_doji,is_bullish,atr
Date,,,,,,,,,,,,
2024-01-01 00:00:00+00:00,100.0,101.1,99.5,100.6,1000000,1.6,0.6,0.375000,True,False,True,1.600000
2024-01-08 00:00:00+00:00,100.6,101.1,99.6,100.1,1000000,1.5,0.5,0.333333,True,False,False,1.592857
2024-01-15 00:00:00+00:00,100.1,101.2,99.6,100.7,1000000,1.6,0.6,0.375000,True,False,True,1.593367
2024-01-22 00:00:00+00:00,100.7,101.2,99.7,100.2,1000000,1.5,0.5,0.333333,True,False,False,1.586698
2024-01-29 00:00:00+00:00,100.2,101.3,99.7,100.8,1000000,1.6,0.6,0.375000,True,False,True,1.587648
2024-02-05 00:00:00+00:00,100.8,101.3,99.8,100.3,1000000,1.5,0.5,0.333333,True,False,False,1.581388
2024-02-12 00:00:00+00:00,100.3,101.4,99.8,100.9,1000000,1.6,0.6,0.375000,True,False,True,1.582717
2024-02-19 00:00:00+00:00,100.9,101.4,99.9,100.4,1000000,1.5,0.5,0.333333,True,False,False,1.576809
2024-02-26 00:00:00+00:00,100.4,101.5,99.9,101.0,1000000,1.6,0.6,0.375000,True,False,True,1.578465


In [6]:
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "notebook_connected"

COLOR_BULL = "#26a69a"
COLOR_BEAR = "#ef5350"
COLOR_ATR  = "#7c83fd"
BG         = "#131722"
GRID       = "#1e222d"

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.65, 0.35],
    vertical_spacing=0.03,
)

# ── candles ────────────────────────────────────────────────────────────────
fig.add_trace(go.Candlestick(
    x=df.index,
    open=df["open"], high=df["high"],
    low=df["low"],   close=df["close"],
    increasing_line_color=COLOR_BULL,
    decreasing_line_color=COLOR_BEAR,
    name="Price",
), row=1, col=1)

# ── ATR line ───────────────────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=df.index,
    y=df["atr"],
    mode="lines",
    line=dict(color=COLOR_ATR, width=1.5),
    fill="tozeroy",
    fillcolor="rgba(124,131,253,0.10)",
    name=f"ATR({n})",
    hovertemplate="%{x}<br>ATR: %{y:.3f}<extra></extra>",
), row=2, col=1)

# ── layout ─────────────────────────────────────────────────────────────────
fig.update_layout(
    title=f"ATR({n}) — Average True Range",
    height=580,
    plot_bgcolor=BG,
    paper_bgcolor=BG,
    font_color="white",
    xaxis_rangeslider_visible=False,
    hovermode="x unified",
    legend=dict(orientation="h", y=1.04),
    bargap=0.15,
)

shared_axis = dict(gridcolor=GRID, showgrid=True, zeroline=False, linecolor=GRID)
fig.update_xaxes(**shared_axis)
fig.update_yaxes(**shared_axis)

fig.update_yaxes(title_text="Price", row=1, col=1)
fig.update_yaxes(title_text=f"ATR({n})", row=2, col=1)
fig.update_xaxes(showticklabels=True, row=2, col=1)

fig.show()
